Notebook to compute steric sea level anomaly plots.

In [ ]:
import numpy as np
import xarray as xr
import xesmf as xe

# modules for plotting datetime data
import matplotlib.dates as mdates
from matplotlib.axis import Axis

# modules for using datetime variables
import datetime
from datetime import time

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm

from xgcm import Grid
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import cartopy.crs as ccrs
import cmocean

import subprocess as sp

import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

from matplotlib.ticker import ScalarFormatter

from xclim import ensembles
from xoverturning import calcmoc
import cmip_basins
import momlevel

import cftime
from pandas.errors import OutOfBoundsDatetime  # Import the specific error

import os

import dask

In [ ]:
myVars = globals()

In [ ]:
from matplotlib import font_manager
# Specify the path to your custom .otf font file
font_path = '/home/Kiera.Lowman/.fonts/HelveticaNeueRoman.otf'

# Add the font to the matplotlib font manager
font_manager.fontManager.addfont(font_path)

# Retrieve the font's name from the file
prop = font_manager.FontProperties(fname=font_path)
font_name = prop.get_name()

# Set the default font globally
plt.rcParams['font.family'] = font_name

plt.rcParams['axes.labelsize'] = 12    # Axis label size
plt.rcParams['xtick.labelsize'] = 10     # X-axis tick label size
plt.rcParams['ytick.labelsize'] = 10     # Y-axis tick label size
plt.rcParams['axes.titlesize'] = 14      # Title size
plt.rcParams['legend.fontsize'] = 10     # Legend font size

In [ ]:
%run /home/Kiera.Lowman/Kd-sensitivity-analysis/notebooks/read_and_calculate.ipynb # to get selecting_basins function
%run /home/Kiera.Lowman/Kd-sensitivity-analysis/notebooks/plotting_functions.ipynb
%run /home/Kiera.Lowman/Kd-sensitivity-analysis/notebooks/compute_ensemble_means.ipynb
%run /home/Kiera.Lowman/Kd-sensitivity-analysis/notebooks/sea_level_plot_functions.ipynb
%run /home/Kiera.Lowman/Kd-sensitivity-analysis/notebooks/time_series_plotting.ipynb

# Important variables and paths

In [ ]:
profiles = ['surf','therm','mid','bot']
prof_strings = ["Surf","Therm","Mid","Bot"]

power_inputs = ['0.1TW', '0.2TW', '0.3TW']
power_var_suff = ['0p1TW', '0p2TW', '0p3TW']
power_strings = ['0.1 TW', '0.2 TW', '0.3 TW']

time_chunk = 25
level_variables = ["temp", "salt", "volcello"]

starts = [1, 76, 176]
ends = [25, 100, 200]

ts_dir = "/home/Kiera.Lowman/Python_figures/45deg_70yr_time_series/"

# Global-mean SL anomalies time series

## Read SLA time series data

### Read ensemble means

In [ ]:
for diff_type in ['const','doub-2xctrl','doub-1860exp','doub-1860ctrl']:
    
    for power in power_var_suff:
        SLA_fname = f"/archive/Kiera.Lowman/SLA_mixing_exps/{diff_type}_45deg_70yr_{power}_SLA_1_200.nc"
        
        for prof in profiles:

            if diff_type == 'const':
                ds_name = f'const_{prof}_{power}_1_200_SLA'
            elif diff_type == 'doub-2xctrl':
                ds_name = f'doub_{prof}_{power}_1_200_2xctrl_SLA'
            elif diff_type == 'doub-1860exp':
                ds_name = f'doub_{prof}_{power}_1_200_1860_SLA'
            elif diff_type == 'doub-1860ctrl':
                ds_name = f'doub_{prof}_{power}_1_200_const_ctrl_SLA'
            
            myVars[ds_name] = xr.open_dataset(SLA_fname, group=prof)
            print(f"Done {ds_name}")

SLA_fname = "/archive/Kiera.Lowman/SLA_mixing_exps/doub_ctrl_1_200_SLA.nc"
ds_name = "doub_ctrl_1_200_SLA"
myVars[ds_name] = xr.open_dataset(SLA_fname)
print(f"Done {ds_name}")

### Read ensemble members

In [ ]:
SLA_dir_root = f"/archive/Kiera.Lowman/SLA_mixing_exps/SLA_single_members"
n_ens = 5
mem_offset = 2 # starting with mem2

for diff_type in ['const','doub-2xctrl','doub-1860exp','doub-1860ctrl']:
    for power in power_var_suff:
        for prof in profiles:

            if diff_type == 'const':
                list_name = f'const_{prof}_{power}_1_200_SLA_mem_list'
            elif diff_type == 'doub-2xctrl':
                list_name = f'doub_{prof}_{power}_1_200_2xctrl_SLA_mem_list'
            elif diff_type == 'doub-1860exp':
                list_name = f'doub_{prof}_{power}_1_200_1860_SLA_mem_list'
            elif diff_type == 'doub-1860ctrl':
                list_name = f'doub_{prof}_{power}_1_200_const_ctrl_SLA_mem_list'

            myVars[list_name] = [None] * n_ens

            for ens_idx in range(n_ens):
                SLA_fname = f"{SLA_dir_root}/{diff_type}_45deg_70yr_{power}_SLA_1_200_mem{ens_idx+mem_offset}.nc"
            
                myVars[list_name][ens_idx] = xr.open_dataset(SLA_fname, group=prof)
                
            print(f"Done {list_name}")

## doub control
list_name = "doub_ctrl_1_200_SLA_mem_list"
myVars[list_name] = [None] * n_ens
for ens_idx in range(n_ens):
    SLA_fname = f"{SLA_dir_root}/doub_ctrl_1_200_SLA_mem{ens_idx+mem_offset}.nc"
    myVars[list_name][ens_idx] = xr.open_dataset(SLA_fname)
    
print(f"Done {list_name}")

# Plot time series

In [ ]:
start = 1
end = 200

plot_mixing_SLA_ts('doub',ts_dir,start,end,fig_pref='45deg_70yr',
                   savefig=True,
                   plot_ens_env=True
             )

In [ ]:
start = 1
end = 200

plot_rad_SLA_ts('doub',ts_dir,start,end,fig_pref='45deg_70yr',
                savefig=True,
                plot_ens_env=True
             )

In [ ]:
start = 1
end = 200

plot_mix_pow_SLA_ts('const',ts_dir,start,end,fig_pref='45deg_70yr',
                    savefig=True,
                    plot_ens_env=True
             )

In [ ]:
start = 1
end = 200

plot_mix_pow_SLA_ts('doub',ts_dir,start,end,fig_pref='45deg_70yr',
                    savefig=True,
                    plot_ens_env=True
             )

In [ ]:
start = 1
end = 200

plot_lin_SLA_ts('doub',ts_dir,start,end,fig_pref='45deg_70yr',
                ystep_list = [0.1, 0.1, 0.1],
                savefig=True,
                plot_ens_env=True
             )

In [ ]:
start = 1
end = 200

plot_mixing_SLA_lin_pow('const',ts_dir,start,end,fig_pref='45deg_70yr',
                        omit_ctrl=False,
                        ystep = 0.05,
                        savefig=True,
                        plot_ens_env=True
             )